# Predict Customer Churn - Version 3

## Stacking with Meta-Learner (Ridge)
- **Base Models**: XGBoost + LightGBM + CatBoost + LogisticRegression
- **Meta-Learner**: Ridge Regression (stacking)
- **Training Data**: 601K rows (594K competition + 7K IBM)
- **Optimization**: Optuna (CatBoost tuning, alpha search for Ridge)
- **Output**: V7-V9 submissions

## 1. Libraries & Setup

In [ ]:
import numpy as np
import pandas as pd
import os, warnings, pickle
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', font_scale=1.1)

# Models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression, Ridge

# Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score, roc_curve

# Optuna
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print('✅ All libraries loaded!')

## 2. Load All Datasets

In [ ]:
# Competition data
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')

# Original IBM real data
original = pd.read_csv('/kaggle/input/datasets/thedrzee/customer-churn-in-telecom-sample-dataset-by-ibm/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Previous submissions V1-V6
v1 = pd.read_csv('/kaggle/input/datasets/mohankrishnathalla/predict-customer-churn-submission-dataset/V1.csv')
v2 = pd.read_csv('/kaggle/input/datasets/mohankrishnathalla/predict-customer-churn-submission-dataset/V2.csv')

print(f'Train:    {train.shape}')
print(f'Test:     {test.shape}')
print(f'Original: {original.shape}')

# Fix & combine
original['Churn'] = original['Churn'].map({'Yes': 'Yes', 'No': 'No'})
original['id'] = range(900000, 900000 + len(original))
train_combined = pd.concat([train, original], ignore_index=True)
print(f'Combined: {train_combined.shape}')

## 3. Feature Engineering

In [ ]:
def preprocess_v3(df, is_train=True):
    df = df.copy()
    
    # Drop ID columns
    df.drop(columns=['id', 'customerID'], inplace=True, errors='ignore')
    
    # Fix TotalCharges
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
    
    # Churn target
    if is_train and 'Churn' in df.columns:
        df['Churn'] = (df['Churn'] == 'Yes').astype(int)
    
    # Binary columns
    for col in ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']:
        if col in df.columns:
            df[col] = (df[col] == 'Yes').astype(int)
    df['gender'] = (df['gender'] == 'Male').astype(int)
    
    # Feature engineering (same as V2)
    df['AvgMonthlyCharge'] = df['TotalCharges'] / (df['tenure'].clip(lower=1))
    df['ChargeRatio']      = df['MonthlyCharges'] / (df['AvgMonthlyCharge'].clip(lower=0.01))
    df['IsNewCustomer']    = (df['tenure'] <= 12).astype(int)
    df['IsLongTerm']       = (df['tenure'] >= 60).astype(int)
    df['ChargeXTenure']    = df['MonthlyCharges'] * df['tenure']
    df['IsHighValue']      = (df['MonthlyCharges'] > 70).astype(int)
    
    # Services count
    service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                    'TechSupport', 'StreamingTV', 'StreamingMovies']
    binned = {}
    for col in service_cols:
        if col in df.columns and df[col].dtype == object:
            binned[col + '_bin'] = (df[col] == 'Yes').astype(int)
    
    df['TotalServices'] = df['PhoneService'].copy()
    for col in service_cols:
        if col + '_bin' in binned:
            df['TotalServices'] += binned[col + '_bin']
    
    df['ChargePerService'] = df['MonthlyCharges'] / (df['TotalServices'].clip(lower=1))
    
    # Contract risk
    if 'Contract' in df.columns:
        contract_risk = {'Month-to-month': 3, 'One year': 2, 'Two year': 1}
        df['ContractRisk'] = df['Contract'].map(contract_risk).fillna(2)
        df['TenureContractRisk']  = df['tenure'] * df['ContractRisk']
        df['ChargeContractRisk']  = df['MonthlyCharges'] * df['ContractRisk']
        df['HighRiskFlag'] = ((df['ContractRisk'] == 3) & 
                             (df['IsNewCustomer'] == 1)).astype(int)
    
    # One-hot encode categorical
    ohe_cols = ['MultipleLines', 'InternetService',
                'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies',
                'Contract', 'PaymentMethod']
    ohe_cols = [c for c in ohe_cols if c in df.columns]
    df = pd.get_dummies(df, columns=ohe_cols, drop_first=False, dtype=int)
    
    return df

# Apply preprocessing
train_clean = preprocess_v3(train_combined, is_train=True)
test_clean  = preprocess_v3(test, is_train=False)

# Align columns
feature_cols = [c for c in train_clean.columns if c != 'Churn']
for col in feature_cols:
    if col not in test_clean.columns:
        test_clean[col] = 0
test_clean = test_clean[feature_cols]

print(f'Train clean: {train_clean.shape}')
print(f'Test clean:  {test_clean.shape}')
print(f'Features:    {len(feature_cols)}')

## 4. Cross-Validation Setup

In [ ]:
# Target & features
y = train_clean['Churn']
X = train_clean.drop(columns=['Churn'])

# CV strategy — 5 folds for stability
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f'Training set: {X.shape}')
print(f'Features:     {X.shape[1]}')
print(f'Churn rate:   {y.mean():.2%}')

# Class imbalance
churn_ratio      = y.mean()
scale_pos_weight = (1 - churn_ratio) / churn_ratio
print(f'Scale pos weight: {scale_pos_weight:.2f}')

## 5. Optuna Hyperparameter Tuning — CatBoost

In [ ]:
# Pre-tuned XGBoost & LightGBM params
best_xgb_params = {
    'n_estimators': 967, 'max_depth': 5, 'learning_rate': 0.05002,
    'subsample': 0.7716, 'colsample_bytree': 0.6621,
    'min_child_weight': 9, 'reg_alpha': 5.67e-06, 'reg_lambda': 1.167,
    'gamma': 0.5593
}

best_lgbm_params = {
    'n_estimators': 1049, 'max_depth': 7, 'learning_rate': 0.0271,
    'num_leaves': 127, 'min_child_samples': 14, 'subsample': 0.9054,
    'bagging_freq': 1, 'colsample_bytree': 0.9506,
    'reg_alpha': 0.5099, 'reg_lambda': 0.0144
}

print(' XGBoost & LightGBM params pre-loaded')

def objective_catboost(trial):
    params = {
        'iterations':           trial.suggest_int('iterations', 300, 2000),
        'depth':                trial.suggest_int('depth', 3, 10),
        'learning_rate':        trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg':          trial.suggest_float('l2_leaf_reg', 0.1, 10.0, log=True),
        'bagging_temperature':  trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength':      trial.suggest_float('random_strength', 0.0, 1.0),
        'border_count':         trial.suggest_int('border_count', 32, 254),
    }
    
    model = CatBoostClassifier(
        **params,
        loss_function='Logloss',
        auto_class_weights='Balanced',
        task_type='GPU',
        random_state=42,
        verbose=False
    )
    
    scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=1)
    return scores.mean()

print(' Tuning CatBoost with Optuna (20 trials)...')
study_catboost = optuna.create_study(direction='maximize')
study_catboost.optimize(objective_catboost, n_trials=20, show_progress_bar=True)
print(f' CatBoost best CV ROC-AUC: {study_catboost.best_value:.5f}')

## 6. Out-Of-Fold Predictions Generation

In [ ]:
print(' Generating OOF predictions for 4 base models...')

oof_xgb   = np.zeros(len(X))
oof_lgbm  = np.zeros(len(X))
oof_cb    = np.zeros(len(X))
oof_lr    = np.zeros(len(X))
test_xgb  = np.zeros(len(test_clean))
test_lgbm = np.zeros(len(test_clean))
test_cb   = np.zeros(len(test_clean))
test_lr   = np.zeros(len(test_clean))

fold_aucs = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), 1):
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    # XGBoost
    xgb = XGBClassifier(**best_xgb_params, scale_pos_weight=scale_pos_weight,
                       objective='binary:logistic', eval_metric='auc',
                       tree_method='hist', device='cuda', random_state=42, verbosity=0)
    xgb.fit(X_tr, y_tr)
    oof_xgb[val_idx] = xgb.predict_proba(X_va)[:, 1]
    test_xgb += xgb.predict_proba(test_clean)[:, 1] / 5
    
    # LightGBM
    lgbm = LGBMClassifier(**best_lgbm_params, objective='binary', metric='auc',
                         device='cpu', n_jobs=-1, is_unbalance=True,
                         random_state=42, verbose=-1)
    lgbm.fit(X_tr, y_tr)
    oof_lgbm[val_idx] = lgbm.predict_proba(X_va)[:, 1]
    test_lgbm += lgbm.predict_proba(test_clean)[:, 1] / 5
    
    # CatBoost
    cb = CatBoostClassifier(**study_catboost.best_params, loss_function='Logloss',
                           auto_class_weights='Balanced', task_type='GPU',
                           random_state=42, verbose=False)
    cb.fit(X_tr, y_tr)
    oof_cb[val_idx] = cb.predict_proba(X_va)[:, 1]
    test_cb += cb.predict_proba(test_clean)[:, 1] / 5
    
    # Logistic Regression (scaled)
    scale_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
    scaler = StandardScaler()
    X_tr_scaled = X_tr.copy()
    X_va_scaled = X_va.copy()
    X_tr_scaled[scale_cols] = scaler.fit_transform(X_tr[scale_cols])
    X_va_scaled[scale_cols] = scaler.transform(X_va[scale_cols])
    
    lr = LogisticRegression(random_state=42, max_iter=1000, C=1.0)
    lr.fit(X_tr_scaled, y_tr)
    oof_lr[val_idx] = lr.predict_proba(X_va_scaled)[:, 1]
    test_lr += lr.predict_proba(scaler.transform(test_clean[scale_cols]))[:, 1] / 5
    
    fold_auc = roc_auc_score(y_va, oof_xgb[val_idx])
    fold_aucs.append(fold_auc)
    print(f'  Fold {fold}/5 done — XGB OOF AUC: {fold_auc:.5f}')

print(f'\nOOF AUCs (XGB): mean={np.mean(fold_aucs):.5f}  std={np.std(fold_aucs):.5f}')

## 7. Meta-Learner Stacking (Ridge)

In [ ]:
# Stack OOF predictions
oof_stack = np.column_stack([oof_xgb, oof_lgbm, oof_cb, oof_lr])
test_stack = np.column_stack([test_xgb, test_lgbm, test_cb, test_lr])

print(f'OOF stack shape: {oof_stack.shape}')
print(f'Test stack shape: {test_stack.shape}')

# Find best alpha for Ridge meta-learner
best_alpha = 1.0
best_alpha_score = 0
alphas_tested = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

print('\n Testing Ridge alpha values...')
for alpha in alphas_tested:
    ridge_meta = Ridge(alpha=alpha, random_state=42)
    ridge_meta.fit(oof_stack, y)
    ridge_preds = ridge_meta.predict(oof_stack)
    ridge_auc = roc_auc_score(y, ridge_preds)
    print(f'   alpha={alpha:7.3f} → OOF AUC: {ridge_auc:.5f}')
    if ridge_auc > best_alpha_score:
        best_alpha = alpha
        best_alpha_score = ridge_auc

print(f'\nBest Ridge alpha: {best_alpha} (AUC: {best_alpha_score:.5f})')

# Train final Ridge meta-learner
meta_model = Ridge(alpha=best_alpha, random_state=42)
meta_model.fit(oof_stack, y)
meta_oof_preds = meta_model.predict(oof_stack)
meta_test_preds = meta_model.predict(test_stack)

# Clip predictions
meta_oof_preds = np.clip(meta_oof_preds, 0, 1)
meta_test_preds = np.clip(meta_test_preds, 0, 1)

meta_oof_auc = roc_auc_score(y, meta_oof_preds)
print(f'Meta-learner OOF ROC-AUC: {meta_oof_auc:.5f}')

## 8. Full Data Refit with Multiple Seeds

In [ ]:
# Multi-seed averaging for stability
seeds = [42, 123, 456, 789, 1024, 2024, 3141, 5678, 9999, 7777]

test_final = np.zeros((len(test_clean), len(seeds)))

print('Refitting base models with multiple seeds...')
for i, seed in enumerate(seeds, 1):
    # XGBoost
    xgb_full = XGBClassifier(**best_xgb_params, scale_pos_weight=scale_pos_weight,
                            objective='binary:logistic', eval_metric='auc',
                            tree_method='hist', device='cuda',
                            random_state=seed, verbosity=0)
    xgb_full.fit(X, y)
    xgb_test_i = xgb_full.predict_proba(test_clean)[:, 1]
    
    # LightGBM
    lgbm_full = LGBMClassifier(**best_lgbm_params, objective='binary', metric='auc',
                              device='cpu', n_jobs=-1, is_unbalance=True,
                              random_state=seed, verbose=-1)
    lgbm_full.fit(X, y)
    lgbm_test_i = lgbm_full.predict_proba(test_clean)[:, 1]
    
    # CatBoost
    cb_full = CatBoostClassifier(**study_catboost.best_params, loss_function='Logloss',
                                auto_class_weights='Balanced', task_type='GPU',
                                random_state=seed, verbose=False)
    cb_full.fit(X, y)
    cb_test_i = cb_full.predict_proba(test_clean)[:, 1]
    
    # LogReg (scaled)
    scaler = StandardScaler()
    X_scaled = X.copy()
    test_scaled = test_clean.copy()
    X_scaled[scale_cols] = scaler.fit_transform(X[scale_cols])
    test_scaled[scale_cols] = scaler.transform(test_clean[scale_cols])
    
    lr_full = LogisticRegression(random_state=seed, max_iter=1000, C=1.0)
    lr_full.fit(X_scaled, y)
    lr_test_i = lr_full.predict_proba(test_scaled)[:, 1]
    
    # Stack & meta-predict
    test_stack_i = np.column_stack([xgb_test_i, lgbm_test_i, cb_test_i, lr_test_i])
    test_final[:, i-1] = meta_model.predict(test_stack_i)
    
    print(f'  Seed {seed}: done')

# Average across seeds
test_ensemble = test_final.mean(axis=1)
test_ensemble = np.clip(test_ensemble, 0, 1)
print(f'\nTest ensemble mean: {test_ensemble.mean():.4f}')

## 9. Submission Generation

In [ ]:
# Load previous submissions for blending
v1 = pd.read_csv('/kaggle/input/datasets/mohankrishnathalla/predict-customer-churn-submission-dataset/V1.csv')
v2 = pd.read_csv('/kaggle/input/datasets/mohankrishnathalla/predict-customer-churn-submission-dataset/V2.csv')

# Align with test set
v1_probs = v1.set_index('id').reindex(test['id'])['Churn'].values
v2_probs = v2.set_index('id').reindex(test['id'])['Churn'].values

# V7: Meta-stack only
sub_v7 = pd.DataFrame({'id': test['id'], 'Churn': test_ensemble})
sub_v7.to_csv('submission_v7_meta_stack.csv', index=False)
print(' submission_v7_meta_stack.csv')

# V8: Weighted blend
blend_v8 = 0.6 * test_ensemble + 0.4 * v1_probs
blend_v8 = np.clip(blend_v8, 0, 1)
sub_v8 = pd.DataFrame({'id': test['id'], 'Churn': blend_v8})
sub_v8.to_csv('submission_v8_blend.csv', index=False)
print(' submission_v8_blend.csv')

# V9: Rank-based blend
from scipy.stats import rankdata
def rank_blend(*arrays, weights):
    ranked = [rankdata(a) / len(a) for a in arrays]
    return sum(w * r for w, r in zip(weights, ranked))

blend_v9 = rank_blend(test_ensemble, v1_probs, weights=[0.7, 0.3])
sub_v9 = pd.DataFrame({'id': test['id'], 'Churn': blend_v9})
sub_v9.to_csv('submission_v9_rank.csv', index=False)
print(' submission_v9_rank.csv')

## 10. CV-LB Tracking

In [ ]:
# Summary table
results = pd.DataFrame({
    'Version': ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9'],
    'Method': ['XGB baseline', 'LogReg baseline', 'Ensemble blend', 
               'Ensemble+blend', 'Blend all', 'Rank blend',
               'Meta-stack', 'Meta+blend', 'Meta+rank'],
    'CV Score': [0.91689, 'N/A', 0.91594, 0.91501, 'N/A', 'N/A',
                 f'{meta_oof_auc:.5f}', 'N/A', 'N/A'],
    'Public LB': [0.91282, 'TBD', 'TBD', 'TBD', 'TBD', 'TBD',
                  'TBD', 'TBD', 'TBD'],
    'Status': ['☑', 'baseline', 'V4', 'V4', 'V4', 'V4',
               '← ACTIVE', 'blend', 'stable']
})
print(results.to_string(index=False))

## 11. ROC Curves & Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ROC curves
for name, probs, color in [
    ('XGBoost OOF',   oof_xgb,   '#4E79A7'),
    ('LightGBM OOF',  oof_lgbm,  '#E15759'),
    ('CatBoost OOF',  oof_cb,    '#59A14F'),
    ('LogReg OOF',    oof_lr,    '#F28E2B'),
    ('Meta-Stack',    meta_oof_preds, '#B07AA1'),
]:
    fpr, tpr, _ = roc_curve(y, probs)
    auc = roc_auc_score(y, probs)
    axes[0].plot(fpr, tpr, lw=2.5, color=color,
                label=f'{name} (AUC={auc:.5f})')

axes[0].plot([0,1],[0,1],'k--',lw=1.5,label='Random')
axes[0].set_xlabel('False Positive Rate', fontsize=11)
axes[0].set_ylabel('True Positive Rate', fontsize=11)
axes[0].set_title('ROC Curves — All Models', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# Distributions
axes[1].hist(test_ensemble, bins=50, color='#B07AA1', alpha=0.7, label='V7: Meta-stack')
axes[1].hist(blend_v8, bins=50, color='#59A14F', alpha=0.6, label='V8: Blend')
axes[1].hist(blend_v9, bins=50, color='#E15759', alpha=0.5, label='V9: Rank')
axes[1].axvline(0.5, color='black', lw=2, linestyle='--')
axes[1].set_xlabel('Churn Probability', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].set_title('Prediction Distributions', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Final Summary

In [ ]:
print('='*70)
print('VERSION 3 — META-LEARNER STACKING (RIDGE) — FINAL SUMMARY')
print('='*70)
print(f'\nTraining Data:')
print(f'  Rows:      {len(X):,}')
print(f'  Features:  {X.shape[1]}')
print(f'  Churn %:   {y.mean()*100:.1f}%')
print(f'\nBase Models OOF AUC:')
print(f'  XGBoost:   {roc_auc_score(y, oof_xgb):.5f} (pre-tuned)')
print(f'  LightGBM:  {roc_auc_score(y, oof_lgbm):.5f} (pre-tuned)')
print(f'  CatBoost:  {roc_auc_score(y, oof_cb):.5f} (Optuna 20 trials)')
print(f'  LogReg:    {roc_auc_score(y, oof_lr):.5f}')
print(f'\nMeta-Learner Performance:')
print(f'  Model:     Ridge(alpha={best_alpha})')
print(f'  OOF AUC:   {meta_oof_auc:.5f}')
print(f'\nTest Set Predictions:')
print(f'  V7 (Meta-stack): mean={test_ensemble.mean():.4f}')
print(f'  V8 (Blend):      mean={blend_v8.mean():.4f}')
print(f'  V9 (Rank):       mean={blend_v9.mean():.4f}')
print(f'\nSubmission Strategy:')
print(f'  1. Submit V7 (meta-stack) — most optimized')
print(f'  2. Compare V7 public LB vs V9 (rank blend)')
print(f'  3. Choose stable approach (V7 or V9)')
print(f'\nNote: Ridge meta-learner shows NaN in CV scoring')
print(f'      despite valid OOF predictions. See V4 for LogReg fix.')
print('='*70)